In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

test_df = pd.read_csv("../data/test_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
# dense_index = DenseIndex(model, "../data/processed/_dense_court_removed_citation", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)
# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True)

In [5]:
test_df = pd.read_csv("../data/test_rewrite_001.csv")
test_id_2_query_d = {}
for query_id, query in zip(test_df['query_id'], test_df['query']):
    test_id_2_query_d[query_id] = query

In [6]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm
import bge_utils
import numpy as np

test_multi_question_df = pd.read_csv("../data/test_rewrite_002.csv")

recall_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(test_multi_question_df['query_id'], test_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in test_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

query_id_2_recall_hits = {}

recall_hits_l = []

cc_repeat_count = defaultdict(int)

for query_id, query_list in query_id_2_query_list.items():
    
    _hits = []
    for q in query_list:
        hits1 = dense_index.search_with_score(q, top_k=100)
        for hit,_ in hits1:
            cc_repeat_count[hit['citation']] += 1
        hits2 = sparse_index.search_with_score(q, top_k=100)
        for hit,_ in hits2:
            cc_repeat_count[hit['citation']] += 1
        _hits = hits_utils.merge_hits_with_score_l_by_max(_hits,hits_utils.merge_hits_with_score_l_by_max(hits1, hits2))
    
    raw_query = query_list[-1]             

    all_hits = _hits.copy()
    
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    query_id_2_recall_hits[query_id] = all_hits.copy()
    
#     second_layer = citation_utils.compute_citation_score_with_sentence_pos(query_id_2_recall_hits[query_id], decay="log")

#     citations = []
#     for citation, score in second_layer:
#         if citation in court_consideration_d:
#             citations.append(citation)
#         if citation in law_d:
#             citations.append(citation)

#     recall_citations_l.append(list(set(citations)))
    
# r = metric_utils.cal_recall(recall_citations_l, gold_citations_l)
# print(r)

# 准备好document
# recall_hits_l = []
# for recall_citations in recall_citations_l:
#     recall_hits = []
#     for citation in set(recall_citations):
#         if citation in court_consideration_d:
#             recall_hits.append({'citation':citation, 'text':court_consideration_d[citation]})
#         elif citation in law_d:
#             recall_hits.append({'citation':citation, 'text':law_d[citation]})
#     recall_hits_l.append(recall_hits)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[test_001] hits_merge.len: 515
[test_002] hits_merge.len: 644
[test_003] hits_merge.len: 475
[test_004] hits_merge.len: 394
[test_005] hits_merge.len: 510
[test_006] hits_merge.len: 562
[test_007] hits_merge.len: 254
[test_008] hits_merge.len: 305
[test_009] hits_merge.len: 499
[test_010] hits_merge.len: 414
[test_011] hits_merge.len: 636
[test_012] hits_merge.len: 533
[test_013] hits_merge.len: 735
[test_014] hits_merge.len: 526
[test_015] hits_merge.len: 514
[test_016] hits_merge.len: 455
[test_017] hits_merge.len: 401
[test_018] hits_merge.len: 708
[test_019] hits_merge.len: 389
[test_020] hits_merge.len: 630
[test_021] hits_merge.len: 683
[test_022] hits_merge.len: 401
[test_023] hits_merge.len: 441
[test_024] hits_merge.len: 662
[test_025] hits_merge.len: 527
[test_026] hits_merge.len: 391
[test_027] hits_merge.len: 371
[test_028] hits_merge.len: 388
[test_029] hits_merge.len: 572
[test_030] hits_merge.len: 470
[test_031] hits_merge.len: 339
[test_032] hits_merge.len: 555
[test_03

In [7]:
import numpy as np

def logsumexp(scores):
    scores = np.array(scores)
    m = np.max(scores)
    return m + np.log(np.sum(np.exp(scores - m)))

In [8]:
import numpy as np
import math
from collections import defaultdict

def normalize_and_aggregate_max(cc_results: list[dict], 
                            cap: float = None, 
                            cc_norm_score_threshold: float = None) -> dict[str, float]:
    """
    cc_results: [
        {
            "cc_id": "cc_001",
            "rerank_score": 0.95,
            "citations": [
                {"citation_id": "BGE_123", "first_sentence_idx": 1},
                {"citation_id": "BGE_456", "first_sentence_idx": 10},
            ]
        },
        ...
    ]
    返回: {citation_id: aggregated_score}
    """

    # Step 1: 对rerank scores做min-max归一化
    scores = np.array([cc["rerank_score"] for cc in cc_results])
    score_min, score_max = scores.min(), scores.max()
    
    if score_max - score_min < 1e-9:
        normalized_scores = np.ones(len(scores))
    else:
        normalized_scores = (scores - score_min) / (score_max - score_min)

    # Step 2: 可选 - softmax归一化（更平滑，抑制极端值）
    # temperature = 1.0  # 调小则更集中，调大则更均匀
    # exp_scores = np.exp((scores - scores.max()) / temperature)
    # normalized_scores = exp_scores / exp_scores.sum()

    # Step 3: 计算每个citation的得分并sum pooling
    citation_scores = defaultdict(float)
    citation_counts = defaultdict(int)  # 记录出现次数，用于分析高频citation
    
    for cc, norm_score in zip(cc_results, normalized_scores):
        if cc_norm_score_threshold is not None and norm_score < cc_norm_score_threshold:
            continue
        for citation in cc["citations"]:
            # print ("===>", citation)
            cit_id = citation["citation_id"]
            first_idx = max(citation["first_sentence_idx"], 1)  # 防止除以0
            decay = 1.0 / math.log(1+first_idx)

            contribution = norm_score * decay
            
            if cap is not None:
                contribution = min(norm_score * decay, cap)
                
            citation_counts[cit_id] += 1

            if cit_id not in citation_scores:
                citation_scores[cit_id] = contribution
            elif citation_scores[cit_id] < contribution:
                citation_scores[cit_id] = contribution

    # Step 4: 对高频citation做频率加权（对应你说的第2个信号）
    # 思路：出现在k篇CC中，额外乘以log(1+k)做boost
    for cit_id in citation_scores:
        k = citation_counts[cit_id]
        freq_boost = np.log1p(k)  # log(1+k)，k=1时boost=0.69，k=10时=2.4
        citation_scores[cit_id] *= freq_boost

    # Step 5: 对最终citation分数再做一次归一化，方便后续threshold截断
    if citation_scores:
        vals = np.array(list(citation_scores.values()))
        val_max = vals.max()
        citation_scores = {
            cit_id: score / val_max
            for cit_id, score in citation_scores.items()
        }

    return dict(citation_scores), dict(citation_counts)

In [9]:
import numpy as np
import math
from collections import defaultdict

def normalize_and_aggregate(cc_results: list[dict], 
                            cap: float = None, 
                            cc_norm_score_threshold: float = None,
                            top_k: int = None) -> dict[str, float]:
    """
    cc_results: [
        {
            "cc_id": "cc_001",
            "rerank_score": 0.95,
            "citations": [
                {"citation_id": "BGE_123", "first_sentence_idx": 1},
                {"citation_id": "BGE_456", "first_sentence_idx": 10},
            ]
        },
        ...
    ]
    返回: {citation_id: aggregated_score}
    """

    # Step 1: 对rerank scores做min-max归一化
    scores = np.array([cc["rerank_score"] for cc in cc_results])
    score_min, score_max = scores.min(), scores.max()
    
    if score_max - score_min < 1e-9:
        normalized_scores = np.ones(len(scores))
    else:
        normalized_scores = (scores - score_min) / (score_max - score_min)

    # Step 2: 可选 - softmax归一化（更平滑，抑制极端值）
    # temperature = 1.0  # 调小则更集中，调大则更均匀
    # exp_scores = np.exp((scores - scores.max()) / temperature)
    # normalized_scores = exp_scores / exp_scores.sum()

    # Step 3: 计算每个citation的得分并sum pooling
    citation_scores = defaultdict(float)
    citation_counts = defaultdict(int)  # 记录出现次数，用于分析高频citation
    
    for cc, norm_score in zip(cc_results, normalized_scores):
        if cc_norm_score_threshold is not None and norm_score < cc_norm_score_threshold:
            continue
        for citation in cc["citations"]:
            # print ("===>", citation)
            cit_id = citation["citation_id"]
            first_idx = max(citation["first_sentence_idx"], 1)  # 防止除以0
            decay = 1.0 / math.log(1+first_idx)

            contribution = norm_score * decay
            
            if cap is not None:
                contribution = min(norm_score * decay, cap)
                
            citation_counts[cit_id] += 1

            if top_k is not None: 
                if citation_counts[cit_id] <= top_k:
                    citation_scores[cit_id] += contribution
            else:
                citation_scores[cit_id] += contribution

    # Step 4: 对高频citation做频率加权（对应你说的第2个信号）
    # 思路：出现在k篇CC中，额外乘以log(1+k)做boost
    for cit_id in citation_scores:
        k = citation_counts[cit_id]
        freq_boost = np.log1p(k)  # log(1+k)，k=1时boost=0.69，k=10时=2.4
        citation_scores[cit_id] *= freq_boost

    # Step 5: 对最终citation分数再做一次归一化，方便后续threshold截断
    if citation_scores:
        vals = np.array(list(citation_scores.values()))
        val_max = vals.max()
        citation_scores = {
            cit_id: score / val_max
            for cit_id, score in citation_scores.items()
        }

    return dict(citation_scores), dict(citation_counts)


def select_citations_d(citation_scores: dict, threshold: float = 0.1) -> list[str]:
    """
    根据归一化后的分数截断，threshold需要在dev set上调
    F1最优threshold通常在0.05~0.2之间
    """
    return [
        cit_id for cit_id, score in citation_scores.items()
        if score >= threshold
    ]

def select_citations_l(citation_scores: dict, threshold: float = 0.1) -> list[str]:
    """
    根据归一化后的分数截断，threshold需要在dev set上调
    F1最优threshold通常在0.05~0.2之间
    """
    return [
        cit_id for cit_id, score in citation_scores
        if score >= threshold
    ]

In [10]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

from openai import OpenAI
import deepseek_utils
import bge_utils
import evidence_utils
import numpy as np
import math
import os
import os.path
import pickle

query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):

    recall_hits = query_id_2_recall_hits[query_id]

    print("===>",query_id, ", recall_hits.len:", len(recall_hits))

    raw_query = query_list[-1]

    cc_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, [hit for hit,_ in recall_hits])

    print("===>",query_id, ", cc_with_score_l_raw.len:", len(cc_with_score_l_raw))

    cc_results = []
    for cc, rerank_score in cc_with_score_l_raw:
        d = {}
        d['cc_id'] = cc['citation']
        d['rerank_score'] = rerank_score
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        citations = []
        for citation, first_sentence_idx in parsed_cc['citations']:
            citations.append({"citation_id":citation, "first_sentence_idx":first_sentence_idx})
        d["citations"] = citations

        cc_results.append(d)

    # citation_scores, _ = normalize_and_aggregate(cc_results, cc_norm_score_threshold=0.15)
    citation_scores, _ = normalize_and_aggregate_max(cc_results, cc_norm_score_threshold=0.23)
    # citation_scores = rrf_aggregate(cc_results)

    sorted_citation_l = sorted([(citation,score) for citation, score in citation_scores.items()], key=lambda x:x[1], reverse=True)

    sorted_citation_l = [(citation,score) for citation,score in sorted_citation_l if citation in court_consideration_d or citation in law_d]

    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    print("query_result2:", len(query_result2))
    
    result_citation_l.append(';'.join(query_result2))
    query_id_l.append(query_id)
    
    print(f"{query_id} query_result2.len:", len(query_result2))


result_df = pd.DataFrame({'query_id':query_id_l, 'predicted_citations':result_citation_l})
result_df.to_csv("../output/submission_all.csv", index=False)
    


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  0%|          | 0/40 [00:00<?, ?it/s]

===> test_001 , recall_hits.len: 515


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


===> test_001 , cc_with_score_l_raw.len: 515
query_result2: 38
test_001 query_result2.len: 38
===> test_002 , recall_hits.len: 644
===> test_002 , cc_with_score_l_raw.len: 644
query_result2: 133
test_002 query_result2.len: 133
===> test_003 , recall_hits.len: 475
===> test_003 , cc_with_score_l_raw.len: 475
query_result2: 76
test_003 query_result2.len: 76
===> test_004 , recall_hits.len: 394
===> test_004 , cc_with_score_l_raw.len: 394
query_result2: 107
test_004 query_result2.len: 107
===> test_005 , recall_hits.len: 510
===> test_005 , cc_with_score_l_raw.len: 510
query_result2: 189
test_005 query_result2.len: 189
===> test_006 , recall_hits.len: 562
===> test_006 , cc_with_score_l_raw.len: 562
query_result2: 79
test_006 query_result2.len: 79
===> test_007 , recall_hits.len: 254
===> test_007 , cc_with_score_l_raw.len: 254
query_result2: 48
test_007 query_result2.len: 48
===> test_008 , recall_hits.len: 305
===> test_008 , cc_with_score_l_raw.len: 305
query_result2: 135
test_008 quer

In [17]:
limit=19
predicted_citation_l = []
sub_df = pd.read_csv("../output/submission_all.csv")
for query_id, preds in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    citation_l = preds.split(";")
    predicted_citation_l.append(';'.join(citation_l[:limit]))

new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':predicted_citation_l})
new_sub_df.to_csv("../output/submission2.csv", index=False)

In [12]:
limit=30
cc_limit=10
predicted_citation_l = []
sub_df = pd.read_csv("../output/submission_all.csv")
for query_id, preds in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    cc_l = []
    law_l = []
    
    pred_l = preds.split(';')
    for pred in pred_l:
        if pred in court_consideration_d:
            if len(cc_l) >= cc_limit:
                pass
            else:
                cc_l.append(pred)
    for pred in pred_l:
        if limit <= len(cc_l) + len(law_l):
                break
        if pred in law_d:
            law_l.append(pred)

    citation_l = cc_l.copy()
    citation_l.extend(law_l)

    predicted_citation_l.append(';'.join(citation_l[:limit]))

new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':predicted_citation_l})
new_sub_df.to_csv("../output/submission2.csv", index=False)